# 10 Core Orchestrator Checks

In [1]:
import pathlib
import sys

_root = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(_root))

from src.core.orchestrator import Orchestrator
from src.policies.middleware import DeterministicFirstPolicyMiddleware
from src.runtime.openai_agents_runtime import OpenAIAgentsRuntimeAdapter
from src.schemas.events import RuntimeEventType
from src.schemas.tool_io import RiskTier
from src.tools.executor import DeterministicToolExecutor
from src.tools.registry import ToolDescriptor, ToolRegistry

In [2]:
registry = ToolRegistry()
registry.register(
    ToolDescriptor(
        name="double_value",
        handler=lambda x: x * 2,
        risk_tier=RiskTier.HIGH,
        is_state_changing=True,
    )
)
policy = DeterministicFirstPolicyMiddleware()
orchestrator = Orchestrator(
    runtime_adapter=OpenAIAgentsRuntimeAdapter(),
    policy_middleware=policy,
    tool_executor=DeterministicToolExecutor(registry=registry, policy=policy),
)

context = {
    "run_id": "run_core_nb",
    "job_id": "job_core_nb",
    "task_id": "task_core_nb",
    "agent_id": "agent_core_nb",
    "planned_tool_call": {
        "call_id": "tc_core_nb",
        "tool_name": "double_value",
        "arguments": {"x": 11},
        "risk_tier": "high",
        "is_state_changing": True,
    },
}

events = []
async for event in orchestrator.run_turn("sess_core_nb", "run deterministic", context):
    events.append(event)
    print(event.event_type.value, event.payload)

event_types = [e.event_type for e in events]
assert RuntimeEventType.RUN_COMPLETE in event_types, "Missing RUN_COMPLETE event"
tool_progress_states = [
    e.payload.get("state")
    for e in events
    if e.event_type == RuntimeEventType.TOOL_PROGRESS
]
assert "completed" in tool_progress_states, "Missing completed TOOL_PROGRESS state"
print("PASS: orchestrator deterministic tool path")

tool_progress {'call_id': 'tc_core_nb', 'tool_name': 'double_value', 'state': 'queued', 'tool_status': '', 'error_code': '', 'job_id': '', 'lease_token': '', 'lease_expires_at_epoch': '', 'claim_attempt': ''}
tool_progress {'call_id': 'tc_core_nb', 'tool_name': 'double_value', 'state': 'running', 'tool_status': '', 'error_code': '', 'job_id': '', 'lease_token': '', 'lease_expires_at_epoch': '', 'claim_attempt': ''}
tool_progress {'call_id': 'tc_core_nb', 'tool_name': 'double_value', 'state': 'completed', 'tool_status': 'success', 'error_code': 'None', 'job_id': '', 'lease_token': '', 'lease_expires_at_epoch': '', 'claim_attempt': ''}
output_delta {'text': 'processed 1 tool result(s)'}
run_complete {'status': 'completed', 'tool_results_count': 1, 'provider_id': 'openai'}
PASS: orchestrator deterministic tool path


Troubleshooting: if assertions fail, verify `planned_tool_call` and `tool_name` registration match.